### Cart-pole with custom dynamic controller

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/cartpole_dynamic_controller.ipynb)


**Importing Librairies**

This page uses the toolbox *minilink*.


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
import numpy as np


# Defining the dynamics

Here we load a already defined class from the library including the dynamic equations of the cart-pole, which is a function of the form:

$\dot{x} = f(x,u)$


In [ ]:
##############
# System
##############

from minilink import CartPole
from minilink.core import DynamicController

sys  = CartPole()

sys.inputs["u"].upper_bound[0] = +20
sys.inputs["u"].lower_bound[0] = -20

sys.x0[0] = 0.5
sys.x0[1] = 3.0

x_bar = np.array([0,np.pi,0,0]) # target state


#Defining your control law


In [ ]:
from minilink import linearize
from scipy import linalg


###############################################################################
class LQG_Controller(DynamicController):

    feedback_profile = "output"

    ############################
    def __init__(self, A, B, C, D, Qn, Rn):
        """ """

        super().__init__(n=4)

        # Label
        self.name = "Custom Cart-pole Controller"

        self.add_input_port("y", dim=4)
        self.add_input_port("r", dim=1, nominal_value=0.0)
        self.add_output_port("u", dim=1, function=self.ctl, dependencies=())
        self.add_output_port("z", dim=4, function=self.compute_state)

        # Linear gain matrix

        self.K = np.array([-0.5, 25, -1.0, 5.0])

        # Observer gain matrix

        # Solve Riccati equation for observer
        P = linalg.solve_continuous_are(a=A.T, b=C.T, q=Qn, r=Rn)
        L = np.linalg.solve(Rn.T, (C @ P.T)).T

        self.L = L

        self.A = A  # np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
        self.B = B  # np.array([[0], [0], [0], [1]])
        self.C = C  # np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
        self.D = D  # np.array([[0], [0], [0], [0]])

    #############################
    def ctl(self, x, u, t=0, params=None):
        """Control law of the controller"""

        x_hat = x

        u_cmd = -self.K @ x_hat

        u_cmd = np.array([u_cmd])

        return u_cmd

    ############################
    def f(self, x, u, t=0, params=None):
        """Update law of internal controller states"""

        y = self.get_port_values_from_u(u, "y")
        y = y - np.array([0.0, np.pi, 0.0, 0.0])

        A = self.A
        B = self.B
        C = self.C
        D = self.D
        L = self.L
        K = self.K

        x_hat = x

        u_cmd = -K @ x_hat

        u_cmd = np.array([u_cmd])

        dx_hat = A @ x_hat + B @ u_cmd + L @ (y - C @ x_hat)

        return dx_hat


## Controller


In [ ]:
###############################################################################
sys = CartPole()

x_bar = np.array([0.0, np.pi, 0.0, 0.0])  # Up-right position

ss = linearize(sys, x_bar, np.zeros(1), eps=0.01)

print(ss.A())
print(ss.B())
print(ss.C())
print(ss.D())


Qn = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
Rn = 0.001 * np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

ctl = LQG_Controller(ss.A(), ss.B(), ss.C(), ss.D(), Qn, Rn)


# Stochastic sys

Process noise $w$ and measurement noise $v$ are ordinary plant input ports. Drive them with `WhiteNoise` source blocks, as in the pendulum-with-noise diagram.


In [ ]:
from minilink import CartPoleWithNoisePort, DiagramSystem, WhiteNoise

sys2 = CartPoleWithNoisePort()
sys2.x0[0] = 0.5
sys2.x0[1] = 3.0

process_noise = WhiteNoise(1)
process_noise.params.update({"var": 0.01, "mean": 0.0, "seed": 1})

measurement_noise = WhiteNoise(4)
measurement_noise.params.update({"var": 0.001, "mean": 0.0, "seed": 2})

cl_sys = DiagramSystem()
cl_sys.add_subsystem(ctl, "ctl")
cl_sys.add_subsystem(sys2, "sys")
cl_sys.add_subsystem(process_noise, "process_noise")
cl_sys.add_subsystem(measurement_noise, "measurement_noise")

cl_sys.connect("ctl", "u", "sys", "u")
cl_sys.connect("sys", "y", "ctl", "y")
cl_sys.connect("process_noise", "y", "sys", "w")
cl_sys.connect("measurement_noise", "y", "sys", "v")

cl_sys.plot_diagram()


## Simulation

Try to play with the initial conditions:


In [ ]:
cl_sys.compute_trajectory(tf=10.0)


Real states:


In [ ]:
cl_sys.plot_trajectory(signals=("sys:x", "ctl:u"))


Noisy sensors signal:


In [ ]:
cl_sys.plot_trajectory(signals=("sys:y",))


Estimated filtered states:


In [ ]:
cl_sys.plot_trajectory(signals=("ctl:z",))


In [ ]:
# Animate and display the simulation
cl_sys.animate()
